### Exercise 6-3 - The Power of Abstraction

##### Step 1: Setting Up and Understanding the Problem

We will be using ```pygame``` to setup our game.

First, to install, comment out and run the below line!

In [ ]:
# !pip install pygame-ce

Next, let us import some open-source modules that will help us with this!

In [1]:
import pygame as pg
import json
import math

c:\Users\jameskohby\AppData\Local\anaconda3\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.12.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


We have already pre-designed some classes for our game. Let us import them. The files are located in aux_files.zip, download them from eLearn and ensure that they are placed in the same directory as this Python notebook!

In [2]:
from modules.environment import Environment
from modules.enemy import Enemy
from modules.buttons import Button

Some code to get started... (You do not need to know how they work!)

In [3]:
# initialise pygame
pg.init()

# create a clock
clock = pg.time.Clock()

# load json data
with open("assets/world/universe.tmj") as file:
    full_data = json.load(file)

# create environment
env = Environment(full_data, None)
env.pre_process()

# create the game window
screen = pg.display.set_mode((env.WIDTH + env.SIDE, env.HEIGHT))
pg.display.set_caption("Merge Game")

##### Step 2 and 3: Creating our Turrets

The base Turret class has already been created for you.

In [4]:
import modules.turret as t

In [5]:
class Range1Turret(t.Turret):
    
    original_image = t.range_turret_1_image
    max_upgrade = False

    def __init__(self, pos, env):
        ## insert code here ##
        pass
        
        self.set_range()

    def search_target(self, enemies):
        ## insert code here ##
        pass

    def merge(self, turret, env):
        ## insert code here ##
        pass

class Range2Turret(Range1Turret):

    original_image = t.range_turret_2_image
    max_upgrade = False

    def __init__(self, pos, env):
        ## insert code here ##
        pass

        self.set_range()

    def merge(self, turret, env):
        ## insert code here ##
        pass

class Range3Turret(Range2Turret):

    original_image = t.range_turret_3_image
    max_upgrade = False

    def __init__(self, pos, env):
        ## insert code here ##
        pass

        self.set_range()

    def merge(self, turret, env):
        pass

class Range4Turret(Range3Turret):

    original_image = t.range_turret_4_image
    max_upgrade = True

    def __init__(self, pos, env):
        ## insert code here ##
        pass

        self.set_range()



In [6]:
class Range1Turret(t.Turret):
    
    original_image = t.range_turret_1_image
    max_upgrade = False

    def __init__(self, pos, env):
        super().__init__(pos, env)

        # set turret stats
        self.cooldown = 1500
        self.range = 90
        self.damage = 10
        self.cost = 50

        self.set_range()

    def search_target(self, enemies):
        # finds an enemy to target
        x_dist, y_dist = 0, 0
        for enemy in enemies:
            x_dist = enemy.pos[0] - self.center[0]
            y_dist = enemy.pos[1] - self.center[1]
            distance = math.sqrt(x_dist**2 + y_dist**2)
            if distance <= self.range:
                self.target = enemy
                self.angle = math.degrees(math.atan2(-y_dist, x_dist))
                # damage enemy
                self.target.health -= self.damage
                break

    def merge(self, turret, env):
        turret.kill()
        pos = self.pos
        self.kill()
        return Range2Turret(pos, env)

class Range2Turret(Range1Turret):

    original_image = t.range_turret_2_image
    max_upgrade = False

    def __init__(self, pos, env):
        super().__init__(pos, env)

        # set turret stats
        self.cooldown = 1000
        self.range = 110
        self.damage = 20
        self.cost = 150

        self.set_range()

    def merge(self, turret, env):
        turret.kill()
        pos = self.pos
        self.kill()
        return Range3Turret(pos, env)

class Range3Turret(Range2Turret):

    original_image = t.range_turret_3_image
    max_upgrade = False

    def __init__(self, pos, env):
        super().__init__(pos, env)

        # set turret stats
        self.cooldown = 750
        self.range = 130
        self.damage = 20
        self.cost = 400

        self.set_range()

    def merge(self, turret, env):
        turret.kill()
        pos = self.pos
        self.kill()
        return Range4Turret(pos, env)

class Range4Turret(Range3Turret):

    original_image = t.range_turret_4_image
    max_upgrade = True

    def __init__(self, pos, env):
        super().__init__(pos, env)

        # set turret stats
        self.cooldown = 500
        self.range = 150
        self.damage = 30
        self.cost = 900

        self.set_range()



Final bulk of the code to make this run, and enjoy!

In [7]:
env_image = pg.image.load('assets/world/universe.png').convert_alpha()
enemy_image = pg.image.load('assets/images/enemy.png').convert_alpha()
restart_image = pg.image.load('assets/images/restart.png').convert_alpha()

env.image = env_image

# create turret
def create_range_turret(mouse_position):
    mouse_x, mouse_y = mouse_position[0] // env.TILE_SIZE, mouse_position[1] // env.TILE_SIZE

    # check that there isn't already a turret on the tile
    space_free = True
    can_merge = False
    turret_to_merge = None
    for turret in turrets:
        if (mouse_x, mouse_y) == (turret.pos[0], turret.pos[1]):
            space_free = False
            if env.place_type == turret.__class__ and not env.place_type.max_upgrade:
                can_merge = True
                turret_to_merge = turret
                break
    
    if space_free:
        turret = env.place_type((mouse_x, mouse_y), env)
        if turret.cost <= env.money:
            env.money -= turret.cost
            turrets.add(turret)
        else:
            turret.kill()

    if can_merge:
        turret = env.place_type((mouse_x, mouse_y), env)
        if turret.cost <= env.money:
            env.money -= turret.cost
            new_turret = turret_to_merge.merge(turret, env)
            turrets.add(new_turret)
        else:
            turret.kill()

# draw text
tower_font = pg.font.SysFont("Consolas", 16)
text_font = pg.font.SysFont("Consolas", 24, bold = True)
large_font = pg.font.SysFont("Consolas", 36)

def draw_text(text, font, text_col, pos):
    img = font.render(text, True, text_col)
    screen.blit(img, pos)


# create groups
enemies = pg.sprite.Group()
turrets = pg.sprite.Group()

# create buttons
range1_button = Button((env.WIDTH + 30, 120), Range1Turret.original_image, True, tower_font, "{:>3d}".format(50))
range2_button = Button((env.WIDTH + 30, 220), Range2Turret.original_image, True, tower_font, "{:>3d}".format(150))
range3_button = Button((env.WIDTH + 30, 320), Range3Turret.original_image, True, tower_font, "{:>3d}".format(400))
range4_button = Button((env.WIDTH + 30, 420), Range4Turret.original_image, True, tower_font, "{:>3d}".format(900))

restart_button = Button((310, 300), restart_image, True, tower_font, "")

# begin the game loop
run = True
while run:

    clock.tick(env.FPS)

    screen.fill("lightgrey")
    env.draw(screen)

    if env.life <= 0:
        pg.draw.rect(screen, "dodgerblue", (200, 200, 400, 200), border_radius = 30)
        draw_text("GAME OVER", large_font, "grey0", (310, 230))
        if restart_button.draw(screen):
            env.reset()
            # empty groups
            enemies.empty()
            turrets.empty()
    
    else:

        # update groups
        enemies.update(env)
        turrets.update(enemies)

        # draw groups
        enemies.draw(screen)
        for turret in turrets:
            turret.draw(screen)

        # draw text
        draw_text(f"Money: {env.money}", text_font, "black", (env.WIDTH + 30, 20))
        draw_text(f"Health: {env.life}", text_font, "black", (env.WIDTH + 30, 60))

        # spawm enemies
        if pg.time.get_ticks() - env.last_spawn > env.SPAWN_COOLDOWN:
            enemy = Enemy(env.junctions, enemy_image)
            enemies.add(enemy)
            env.last_spawn = pg.time.get_ticks()

        # draw buttons
        if range1_button.draw(screen):
            if not env.placable:
                env.placable = True
            env.place_type = Range1Turret

        if range2_button.draw(screen):
            if not env.placable:
                env.placable = True
            env.place_type = Range2Turret

        if range3_button.draw(screen):
            if not env.placable:
                env.placable = True
            env.place_type = Range3Turret

        if range4_button.draw(screen):
            if not env.placable:
                env.placable = True
            env.place_type = Range4Turret

        if env.placable:
            cursor_rect = env.place_type.original_image.get_rect()
            cursor_pos = pg.mouse.get_pos()
            cursor_rect.center = cursor_pos

            if cursor_pos[0] <= env.WIDTH:
                screen.blit(env.place_type.original_image, cursor_rect)

            if range1_button.draw(screen) or range2_button.draw(screen) or range3_button.draw(screen) or range4_button.draw(screen):
                env.placable = False
                env.place_type = None

    # event handler
    for event in pg.event.get():

        # exit strategy
        if event.type == pg.QUIT:
            run = False

        # mouse click event
        if event.type == pg.MOUSEBUTTONDOWN and event.button == 1:
            mouse_pos = pg.mouse.get_pos()
            # check that the mouse is within the screen
            if 0 <= mouse_pos[0] < env.WIDTH and 0 <= mouse_pos[1] < env.HEIGHT:
                if env.placable:
                    create_range_turret(mouse_pos)

    # update display
    pg.display.flip()

pg.quit()